<a href="https://colab.research.google.com/github/L-SALTOS-UDLA/practica/blob/main/Correlacion_cinco_variables_calidad_tv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Correlación de cinco variables de producción y calidad

## Resumen
Se analizaron 5.643 registros de la hoja Base maestra de 2024–2025. La producción planificada y la producida presentan una asociación positiva muy alta (r = 0,993). La cantidad de reprocesos se asocia negativamente con el cumplimiento de calidad (r = −0,742). Los datos son sintéticos y las correlaciones no demuestran causalidad.

## Método y preparación
Se calcula Pearson únicamente para las cinco variables solicitadas. El nombre de la variable en el Excel es **Cumplimiento de calidad**. Cada fila es un registro de la Base maestra; no se mezcla con el Resumen diario. Se usan todos los registros, sin imputaciones, agregaciones ni eliminación de valores atípicos. Pearson mide asociación lineal: su rango es de −1 a +1.

Para reproducir, coloque **BD_maestria_produccion_calidad_tv_2024_2025(1).xlsx** junto al notebook. Dependencias: Python 3, pandas, numpy, openpyxl y matplotlib.

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOMBRE_ARCHIVO = "BD_maestria_produccion_calidad_tv_2024_2025(1).xlsx"
archivo = Path(NOMBRE_ARCHIVO)
if not archivo.exists():
    archivo = Path("upload") / NOMBRE_ARCHIVO
if not archivo.exists():
    raise FileNotFoundError("Coloca el Excel junto al notebook o modifica archivo.")
variables = ["Cantidad planificada", "Cantidad producida", "Cantidad de reprocesos", "Cumplimiento de calidad", "Pulgadas"]
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 12, "figure.facecolor": "white"})
print("Fuente:", archivo.name)
print("SHA256:", hashlib.sha256(archivo.read_bytes()).hexdigest())

FileNotFoundError: Coloca el Excel junto al notebook o modifica archivo.

## Datos
### 1 Leer y comprobar las cinco columnas

In [ ]:
datos = pd.read_excel(archivo, sheet_name="Base maestra")
seleccion = datos[variables].copy()
assert all(pd.api.types.is_numeric_dtype(seleccion[c]) for c in variables)
print("Registros:", len(seleccion))
print("Periodo:", datos["Fecha"].min().date(), "a", datos["Fecha"].max().date())
print("Faltantes en las cinco columnas:", int(seleccion.isna().sum().sum()))
print("Filas completas duplicadas en la base:", int(datos.duplicated().sum()))
print("ID duplicados:", int(datos["ID registro"].duplicated().sum()))
print("Nota de la fuente:", pd.read_excel(archivo, sheet_name="Diccionario", header=None).iloc[2, 0])
seleccion.describe().round(4)

FileNotFoundError: [Errno 2] No such file or directory: 'upload/BD_maestria_produccion_calidad_tv_2024_2025(1).xlsx'

## Resultados
### 2 Calcular la matriz de correlación
El redondeo solo afecta a la presentación; los cálculos conservan toda la precisión.

In [ ]:
correlacion = seleccion.corr(method="pearson")
assert correlacion.shape == (5, 5)
assert np.allclose(correlacion, correlacion.T)
assert np.allclose(np.diag(correlacion), 1)
assert not correlacion.isna().any().any()
correlacion.round(4)

,Cantidad planificada,Cantidad producida,Cantidad de reprocesos,Cumplimiento de calidad,Pulgadas
Cantidad planificada,1.0000,0.9930,0.4516,-0.0028,-0.5963
Cantidad producida,0.9930,1.0000,0.4512,0.0010,-0.5940
Cantidad de reprocesos,0.4516,0.4512,1.0000,-0.7422,-0.2804
Cumplimiento de calidad,-0.0028,0.0010,-0.7422,1.0000,0.0210
Pulgadas,-0.5963,-0.5940,-0.2804,0.0210,1.0000


### 3 Graficar la matriz
La escala de color se mantiene entre −1 y +1. Cada celda contiene el coeficiente con tres decimales, para permitir interpretar el signo y la magnitud sin depender solo del color.

In [ ]:
etiquetas_x = ["Cantidad\nplanificada", "Cantidad\nproducida", "Cantidad de\nreprocesos", "Cumplimiento\nde calidad", "Pulgadas"]
fig, ax = plt.subplots(figsize=(9.5, 6.0))
im = ax.imshow(correlacion, cmap="viridis", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(5), etiquetas_x)
ax.set_yticks(range(5), variables)
ax.tick_params(axis="both", length=0, labelsize=11)
for i in range(5):
    for j in range(5):
        valor = correlacion.iloc[i, j]
        ax.text(j, i, f"{valor:.3f}", ha="center", va="center", fontsize=13, color="white" if abs(valor) > 0.65 else "#111827")
ax.set_title("Correlación de Pearson de cinco variables\n5.643 registros | 2024–2025", fontsize=15, pad=16)
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03, label="Coeficiente r")
fig.tight_layout()
plt.show()

NameError: name 'plt' is not defined

### 4 Revisar las diez relaciones únicas

In [ ]:
mascara = np.triu(np.ones((5, 5), dtype=bool), k=1)
pares = correlacion.where(mascara).stack().rename("r").reset_index()
pares.columns = ["Variable 1", "Variable 2", "r"]
pares["Magnitud"] = pares["r"].abs()
pares.sort_values("Magnitud", ascending=False).round(4)

,Variable 1,Variable 2,r,Magnitud
0,Cantidad planificada,Cantidad producida,0.9930,0.9930
7,Cantidad de reprocesos,Cumplimiento de calidad,-0.7422,0.7422
3,Cantidad planificada,Pulgadas,-0.5963,0.5963
6,Cantidad producida,Pulgadas,-0.5940,0.5940
1,Cantidad planificada,Cantidad de reprocesos,0.4516,0.4516
4,Cantidad producida,Cantidad de reprocesos,0.4512,0.4512
8,Cantidad de reprocesos,Pulgadas,-0.2804,0.2804
9,Cumplimiento de calidad,Pulgadas,0.0210,0.0210
2,Cantidad planificada,Cumplimiento de calidad,-0.0028,0.0028
5,Cantidad producida,Cumplimiento de calidad,0.0010,0.0010


### 5 Verificar los resultados principales mediante un cálculo independiente

In [ ]:
r_produccion = np.corrcoef(seleccion["Cantidad planificada"], seleccion["Cantidad producida"])[0, 1]
r_calidad = np.corrcoef(seleccion["Cantidad de reprocesos"], seleccion["Cumplimiento de calidad"])[0, 1]
assert np.isclose(r_produccion, correlacion.loc["Cantidad planificada", "Cantidad producida"])
assert np.isclose(r_calidad, correlacion.loc["Cantidad de reprocesos", "Cumplimiento de calidad"])
print(f"Planificada–producida: r = {r_produccion:.6f}")
print(f"Reprocesos–calidad: r = {r_calidad:.6f}")
print("Verificaciones correctas.")

Planificada–producida: r = 0.992965
Reprocesos–calidad: r = -0.742199
Verificaciones correctas.


## Hallazgos
1. **Planificada y producida: r = 0,993.** Los registros con mayor volumen planificado suelen tener mayor volumen producido. No significa que se alcance el 99,3 % de la planificación.
2. **Reprocesos y calidad: r = −0,742.** Es la asociación más intensa con la calidad entre las cuatro variables seleccionadas. Más reprocesos se asocian con menor cumplimiento; esto no prueba que reprocesar cause menor calidad. Los reprocesos pueden ser una respuesta a defectos previos.
3. **Pulgadas y volúmenes:** r = −0,596 con cantidad planificada y r = −0,594 con cantidad producida. Los tamaños mayores se asocian con menores volúmenes en esta base. Pulgadas solo toma tres tamaños, lo que limita la generalización a otros tamaños.
4. **Volúmenes y reprocesos:** r = 0,452 con cantidad planificada y r = 0,451 con cantidad producida. Un mayor volumen puede acumular más reprocesos, sin implicar una tasa mayor.
5. **Calidad y volumen o tamaño:** r = −0,0028 con planificada, 0,0010 con producida y 0,0210 con pulgadas. Las asociaciones lineales son prácticamente nulas en esta simulación; ello no descarta relaciones por grupos o no lineales.

**Conclusión:** estudiar la cantidad de reprocesos junto con su tasa y la secuencia de detección de defectos. Si se pretende predecir antes del cierre, verificar que los reprocesos estén disponibles en ese momento para evitar fuga de información. No inferir causalidad ni decisiones operativas desde datos sintéticos.